# Compressing Lyα-forest spectra: a deeper autoencoder versus PCA

This notebook compares two lossy representations of a 128-pixel Lyα-forest spectrum:

- **principal component analysis (PCA):** the optimal linear reconstruction for a fixed number of orthogonal coefficients;
- **a deep convolutional autoencoder:** a nonlinear encoder and decoder connected only through an explicit latent bottleneck.

$$
F(x)\longrightarrow \underbrace{z}_{d\ \mathrm{stored\ values}}
\longrightarrow \widehat F(x),
\qquad d\in\{1,2,4,8,16,32,64\}.
$$

Both methods use the same training and held-out test spectra, the same seven compressed dimensions, and the same flux-space RMSE. We first verify that the PCA projection and inverse transform are numerically correct. We then inspect how spectral information returns as the compressed dimension grows, before making the final overlaid RMSE comparison.


## Learning objectives

By the end, students should be able to:

1. distinguish compression from supervised physical-field inversion;
2. derive and verify PCA projection and inverse reconstruction;
3. identify the autoencoder bottleneck as the complete per-spectrum code;
4. understand why a deep autoencoder must not contain skip connections around the bottleneck;
5. compare PCA and autoencoder reconstructions at identical dimensions;
6. interpret an RMSE-versus-dimension curve and its test-simulation scatter; and
7. distinguish per-spectrum code size from shared model or basis storage.


## 1. Imports and configuration

We compress noise-free, instrumentally smoothed flux. Random pixel noise is excluded from the target because the immediate question is how efficiently each method represents the physical spectral structure. A denoising extension is included in the exercises.

Every compressed dimension is a power of two from 1 to 64. Since the original spectrum contains 128 values, these correspond to compression ratios from $128{:}1$ to $2{:}1$.


In [ ]:
from pathlib import Path
import copy
import time

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from scipy import ndimage, special

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

# Reproducible CPU execution keeps the exercise portable across laptops.
np.random.seed(7)
torch.manual_seed(7)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(min(4, max(1, torch.get_num_threads())))
device = torch.device("cpu")

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 180,
        "font.size": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "legend.frameon": False,
        "lines.linewidth": 1.8,
    }
)

BLACK = "#202124"
BLUE = "#2878B5"
ORANGE = "#E07A1F"
PURPLE = "#7451A6"
GREY = "#7A7A7A"
LIGHT_GREY = "#D6D6D6"

data_directory = Path("Sims/CMD_z=2_grid128")
box_size = 25.0
redshift = 2.0
h_camels = 0.6711
Omega_m = 0.30
instrument_fwhm_kms = 10.0

train_skewers_per_box = 64
validation_skewers_per_box = 32
test_skewers_per_box = 16

code_dimensions = [1, 2, 4, 8, 16, 32, 64]
latent_dimensions = code_dimensions.copy()
pca_dimensions = code_dimensions.copy()
reference_latent_dimension = 8

maximum_epochs = 150
early_stopping_patience = 24
batch_size = 64
learning_rate = 1.0e-3

print(f"PyTorch {torch.__version__}; device={device}")
print(f"Compared dimensions: {code_dimensions}")


## 2. Load fields and generate realistic spectra

HI density and temperature generate each spectrum through a continuous Voigt calculation. Simulations 0–17 are used for training, 18–19 for validation, and 20–26 only for the final test. PCA and the autoencoder therefore see identical splits, while test realizations remain genuinely held out.

Hubble flow and thermal broadening are included. Signed peculiar velocities are omitted because the scalar CAMELS velocity-modulus grid does not provide a line-of-sight direction.


In [ ]:
hi_path = data_directory / "Grids_HI_IllustrisTNG_CV_128_z=2.0.npy"
temperature_path = data_directory / "Grids_T_IllustrisTNG_CV_128_z=2.0.npy"

missing_files = [
    path for path in (hi_path, temperature_path)
    if not path.exists()
]
if missing_files:
    formatted_paths = "\n".join(f"  - {path}" for path in missing_files)
    raise FileNotFoundError(f"Missing CAMELS files:\n{formatted_paths}")

hi_boxes = np.load(hi_path, mmap_mode="r")
temperature_boxes = np.load(temperature_path, mmap_mode="r")
if hi_boxes.shape != temperature_boxes.shape or hi_boxes.ndim != 4:
    raise ValueError(
        "Expected matching arrays with shape (simulation, x, y, z); "
        f"received {hi_boxes.shape} and {temperature_boxes.shape}."
    )

number_of_pixels = hi_boxes.shape[1]
if number_of_pixels != 128:
    raise ValueError(
        f"This exercise expects 128-pixel spectra, not {number_of_pixels}."
    )

cell_size = box_size / number_of_pixels
x = (np.arange(number_of_pixels) + 0.5) * cell_size

training_simulations = np.arange(0, 18)
validation_simulations = np.arange(18, 20)
test_simulations = np.arange(20, 27)

MSUN_G = 1.98847e33
MPC_CM = 3.085677581e24
M_H_G = 1.6735575e-24
K_B = 1.380649e-16
C_CMS = 2.99792458e10
C_KMS = 2.99792458e5
LAMBDA_ALPHA_CM = 1215.67e-8
GAMMA_ALPHA = 6.262e8
I_ALPHA = 4.45e-18

H_z = 100 * h_camels * np.sqrt(
    Omega_m * (1 + redshift) ** 3 + 1 - Omega_m
)
distance_Mpc = (x - box_size / 2) / h_camels
z_abs = redshift + H_z * distance_Mpc / C_KMS


def optical_depth(rho_hi_grid, temperature):
    """Calculate Lyα optical depth along one 128-cell skewer."""
    rho_comoving = rho_hi_grid * MSUN_G * h_camels**2 / MPC_CM**3
    n_hi = rho_comoving * (1 + z_abs) ** 3 / M_H_G

    thermal_b = np.sqrt(
        2 * K_B * np.clip(temperature, 10, None) / M_H_G
    )
    nu_alpha = C_CMS / LAMBDA_ALPHA_CM
    damping = GAMMA_ALPHA * C_CMS / (
        4 * np.pi * nu_alpha * thermal_b
    )

    velocity_offset = (
        C_CMS
        * (z_abs[:, None] - z_abs[None, :])
        / (1 + z_abs[None, :])
    )
    profile = np.real(
        special.wofz(
            velocity_offset / thermal_b[:, None]
            + 1j * damping[:, None]
        )
    )

    cell_width_cm = cell_size * MPC_CM / h_camels
    amplitude = (
        C_CMS
        * I_ALPHA
        * cell_width_cm
        * n_hi
        / (np.sqrt(np.pi) * thermal_b * (1 + z_abs))
    )
    return np.sum(amplitude[:, None] * profile, axis=0)


velocity_pixel_width = H_z * (cell_size / h_camels) / (1 + redshift)
instrument_sigma_pixels = instrument_fwhm_kms / (
    2 * np.sqrt(2 * np.log(2)) * velocity_pixel_width
)


def make_flux(rho_hi, temperature):
    """Generate ideal flux and apply a Gaussian instrumental profile."""
    intrinsic_flux = np.exp(-optical_depth(rho_hi, temperature))
    return ndimage.gaussian_filter1d(
        intrinsic_flux,
        sigma=instrument_sigma_pixels,
        mode="wrap",
    )


In [ ]:
def choose_positions(number, rng, include_centre=False):
    """Choose unique transverse (y, z) positions in one simulation box."""
    centre = number_of_pixels // 2
    selected = [(centre, centre)] if include_centre else []

    centre_flat_index = centre * number_of_pixels + centre
    available = np.delete(
        np.arange(number_of_pixels**2),
        centre_flat_index,
    )
    draws = rng.choice(
        available,
        number - len(selected),
        replace=False,
    )
    selected.extend(
        divmod(int(value), number_of_pixels)
        for value in draws
    )
    return selected


def build_flux_dataset(
    simulations,
    skewers_per_box,
    random_seed,
    include_centre=False,
):
    """Generate spectra and retain the simulation label of each skewer."""
    rng = np.random.default_rng(random_seed)
    spectra = []
    labels = []

    for simulation in simulations:
        positions = choose_positions(
            skewers_per_box,
            rng,
            include_centre=include_centre,
        )
        for y_index, z_index in positions:
            rho_hi = hi_boxes[
                simulation, :, y_index, z_index
            ].astype(float)
            temperature = temperature_boxes[
                simulation, :, y_index, z_index
            ].astype(float)
            spectra.append(make_flux(rho_hi, temperature))
            labels.append(simulation)

    return np.asarray(spectra, np.float32), np.asarray(labels)


generation_start = time.perf_counter()
train_flux, train_labels = build_flux_dataset(
    training_simulations,
    train_skewers_per_box,
    random_seed=101,
)
validation_flux, validation_labels = build_flux_dataset(
    validation_simulations,
    validation_skewers_per_box,
    random_seed=202,
)
test_flux, test_labels = build_flux_dataset(
    test_simulations,
    test_skewers_per_box,
    random_seed=303,
    include_centre=True,
)

print("train / validation / test:")
print(train_flux.shape, validation_flux.shape, test_flux.shape)
print(f"generation time = {time.perf_counter() - generation_start:.1f} s")


In [ ]:
representative = 0

fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
ax.step(
    x,
    test_flux[representative],
    where="mid",
    color=BLACK,
    linewidth=1.2,
)
ax.fill_between(
    x,
    test_flux[representative],
    1.0,
    step="mid",
    color=BLUE,
    alpha=0.12,
)
ax.set(
    xlabel=r"Distance $x$ [$h^{-1}$ cMpc]",
    ylabel="Transmitted flux",
    title=(
        "Held-out spectrum to compress • "
        f"simulation {test_labels[representative]}"
    ),
    ylim=(-0.03, 1.05),
    xlim=(0, box_size),
)
ax.grid(axis="x", alpha=0.2)
plt.show()


## 3. PCA: an exact linear benchmark

PCA is fitted **only on the training spectra in physical flux units**. If $\mu$ is the training-set mean spectrum and the rows of $V$ are the right-singular vectors of the centred training matrix, then

$$
z_d=(F-\mu)V_d^{\mathsf T},
\qquad
\widehat F_d=\mu+z_dV_d.
$$

This is the correct truncated-SVD reconstruction. It minimizes squared reconstruction error over all rank-$d$ linear subspaces. Fitting PCA after a single global scalar standardization would produce the same directions; using physical flux here simply makes the inverse transform and the final RMSE easier to interpret.


In [ ]:
# Use float64 during the decomposition so the numerical checks below are
# sensitive to implementation mistakes rather than round-off.
train_flux_64 = train_flux.astype(np.float64)
pca_mean_spectrum = train_flux_64.mean(axis=0)
centred_train_flux = train_flux_64 - pca_mean_spectrum

_, singular_values, pca_components = np.linalg.svd(
    centred_train_flux,
    full_matrices=False,
)


def pca_encode(spectra, number_of_components):
    """Project spectra onto the leading training-set PCA components."""
    basis = pca_components[:number_of_components]
    centred_spectra = np.asarray(spectra, dtype=np.float64) - pca_mean_spectrum
    return centred_spectra @ basis.T


def pca_decode(coefficients, number_of_components):
    """Map PCA coefficients back to 128-pixel flux spectra."""
    basis = pca_components[:number_of_components]
    return pca_mean_spectrum + np.asarray(coefficients) @ basis


def pca_reconstruct(spectra, number_of_components):
    coefficients = pca_encode(spectra, number_of_components)
    return pca_decode(coefficients, number_of_components)


explained_variance_fraction = singular_values**2 / np.sum(singular_values**2)
cumulative_variance_fraction = np.cumsum(explained_variance_fraction)

# Three explicit correctness checks.
component_gram_matrix = pca_components @ pca_components.T
orthonormality_error = np.max(
    np.abs(component_gram_matrix - np.eye(number_of_pixels))
)
full_rank_reconstruction = pca_reconstruct(train_flux, number_of_pixels)
full_rank_rmse = np.sqrt(
    np.mean((full_rank_reconstruction - train_flux_64) ** 2)
)
training_rmse_by_dimension = np.array(
    [
        np.sqrt(
            np.mean(
                (pca_reconstruct(train_flux, dimension) - train_flux_64) ** 2
            )
        )
        for dimension in code_dimensions
    ]
)
error_is_monotonic = np.all(np.diff(training_rmse_by_dimension) <= 1e-12)

print(f"Maximum |V V^T - I|       : {orthonormality_error:.3e}")
print(f"Full-rank training RMSE    : {full_rank_rmse:.3e}")
print(f"Truncated error monotonic  : {error_is_monotonic}")
print(
    "Variance explained by 8 PCs: "
    f"{cumulative_variance_fraction[7]:.3f}"
)

assert orthonormality_error < 1e-10
assert full_rank_rmse < 1e-10
assert error_is_monotonic


### 3.1 What PCA captures as components are added

The cumulative explained-variance curve must increase monotonically and reach unity at full rank. The marked points are the seven dimensions used in the PCA–autoencoder comparison. Explained variance is a training-set diagnostic; held-out RMSE remains the final measure of compression quality.


In [ ]:
component_count = np.arange(1, number_of_pixels + 1)

fig, ax = plt.subplots(figsize=(8.6, 4.8), constrained_layout=True)
ax.plot(
    component_count,
    cumulative_variance_fraction,
    color=BLUE,
    linewidth=2.2,
    label="Cumulative explained variance",
)
ax.scatter(
    code_dimensions,
    cumulative_variance_fraction[np.array(code_dimensions) - 1],
    color=BLUE,
    edgecolor="white",
    linewidth=0.8,
    s=55,
    zorder=3,
    label="Compared dimensions",
)
for dimension in code_dimensions:
    fraction = cumulative_variance_fraction[dimension - 1]
    ax.annotate(
        f"{fraction:.2f}",
        (dimension, fraction),
        xytext=(0, 7),
        textcoords="offset points",
        ha="center",
        fontsize=8,
    )
ax.set(
    xscale="log",
    xlabel="Number of PCA components",
    ylabel="Cumulative explained-variance fraction",
    title="PCA progressively recovers the variance of the training spectra",
    xlim=(0.8, 140),
    ylim=(0, 1.04),
)
ax.set_xticks(code_dimensions, labels=[str(value) for value in code_dimensions])
ax.grid(alpha=0.22, which="both")
ax.legend(loc="lower right")
plt.show()


## 4. A deeper nonlinear convolutional autoencoder

The original network used only two convolutional encoder stages. Here the encoder contains **four resolution stages**, and every stage contains two convolutions. The decoder mirrors this depth:

| Stage | Spectral length | Channels |
|---|---:|---:|
| Input | 128 | 1 |
| Encoder block 1 → average pool | 128 → 64 | 8 |
| Encoder block 2 → average pool | 64 → 32 | 16 |
| Encoder block 3 → average pool | 32 → 16 | 32 |
| Encoder block 4 → average pool | 16 → 8 | 48 |
| Dense bottleneck | — | $d$ |
| Decoder | 8 → 16 → 32 → 64 → 128 | 32 → 16 → 8 → 8 |
| Output | 128 | 1 |

Each convolutional block uses two circularly padded convolutions and GELU activations. There are deliberately **no skip connections**: every reconstructed pixel must be generated from the $d$ stored latent numbers.


In [ ]:
class ConvolutionBlock(nn.Module):
    """Two same-length convolutions with smooth nonlinear activations."""

    def __init__(self, input_channels, output_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv1d(
                input_channels,
                output_channels,
                kernel_size=5,
                padding=2,
                padding_mode="circular",
            ),
            nn.GELU(),
            nn.Conv1d(
                output_channels,
                output_channels,
                kernel_size=3,
                padding=1,
                padding_mode="circular",
            ),
            nn.GELU(),
        )

    def forward(self, values):
        return self.layers(values)


class SpectralAutoencoder(nn.Module):
    """Deep 1D autoencoder with a strict d-dimensional bottleneck."""

    def __init__(self, latent_dimension):
        super().__init__()
        self.latent_dimension = latent_dimension

        self.encoder_blocks = nn.ModuleList(
            [
                ConvolutionBlock(1, 8),
                ConvolutionBlock(8, 16),
                ConvolutionBlock(16, 32),
                ConvolutionBlock(32, 48),
            ]
        )
        self.downsample = nn.AvgPool1d(kernel_size=2)
        self.to_latent = nn.Linear(48 * 8, latent_dimension)

        self.from_latent = nn.Linear(latent_dimension, 48 * 8)
        self.decoder_blocks = nn.ModuleList(
            [
                ConvolutionBlock(48, 32),
                ConvolutionBlock(32, 16),
                ConvolutionBlock(16, 8),
                ConvolutionBlock(8, 8),
            ]
        )
        self.output_layer = nn.Conv1d(
            8,
            1,
            kernel_size=3,
            padding=1,
            padding_mode="circular",
        )

    def encode(self, spectra):
        features = spectra
        for block in self.encoder_blocks:
            features = block(features)
            features = self.downsample(features)
        return self.to_latent(features.flatten(start_dim=1))

    def decode(self, latent):
        features = self.from_latent(latent).reshape(-1, 48, 8)
        for block in self.decoder_blocks:
            features = F.interpolate(
                features,
                scale_factor=2,
                mode="linear",
                align_corners=False,
            )
            features = block(features)
        return self.output_layer(features)

    def forward(self, spectra):
        latent = self.encode(spectra)
        return self.decode(latent)


In [ ]:
def architecture_box(ax, x_position, text, colour, width=2.1):
    patch = FancyBboxPatch(
        (x_position - width / 2, 0.65),
        width,
        0.78,
        boxstyle="round,pad=0.04",
        facecolor=colour,
        edgecolor=BLACK,
        linewidth=1.0,
    )
    ax.add_patch(patch)
    ax.text(x_position, 1.04, text, ha="center", va="center", fontsize=9.5)


fig, ax = plt.subplots(figsize=(13, 3.2), constrained_layout=True)
nodes = [
    (1.2, "Flux\n128 × 1", "#E8EEF2"),
    (4.0, "4 encoder blocks\n8 Conv layers\n128 → 8", "#C5DDED"),
    (6.8, "Latent code $z$\n$d$ values", "#E5DCF2"),
    (9.6, "4 decoder blocks\n8 Conv layers\n8 → 128", "#F4D8B8"),
    (12.4, "Reconstruction\n128 × 1", "#DCEEDC"),
]
for x_position, label, colour in nodes:
    architecture_box(ax, x_position, label, colour)
for left, right in zip(nodes[:-1], nodes[1:]):
    ax.add_patch(
        FancyArrowPatch(
            (left[0] + 1.08, 1.04),
            (right[0] - 1.08, 1.04),
            arrowstyle="-|>",
            mutation_scale=13,
            color=GREY,
        )
    )
ax.text(
    6.8,
    1.95,
    "only information path",
    color=PURPLE,
    ha="center",
    fontweight="bold",
)
ax.set_xlim(0, 13.6)
ax.set_ylim(0.25, 2.35)
ax.axis("off")
ax.set_title("Deep autoencoder with a strict low-dimensional bottleneck")
plt.show()


## 5. Train one autoencoder for each bottleneck dimension

Each bottleneck size is trained independently. Model selection uses validation MSE; the held-out test simulations are never used for gradient updates, learning-rate decisions, or epoch selection. Early stopping keeps laptop runtime manageable while allowing the deeper network to converge.


In [ ]:
# A single global affine scaling preserves relative flux amplitudes while
# giving the neural optimizer values of order unity.
flux_mean = float(train_flux.mean())
flux_std = float(train_flux.std())

train_standard = (train_flux - flux_mean) / flux_std
validation_standard = (validation_flux - flux_mean) / flux_std
test_standard = (test_flux - flux_mean) / flux_std


def spectrum_tensor(array):
    """Convert (spectra, pixels) to PyTorch (spectra, channels, pixels)."""
    return torch.as_tensor(array[:, None, :], dtype=torch.float32)


training_set = TensorDataset(spectrum_tensor(train_standard))
validation_set = TensorDataset(spectrum_tensor(validation_standard))
validation_loader = DataLoader(
    validation_set,
    batch_size=128,
    shuffle=False,
    num_workers=0,
)


def train_autoencoder(latent_dimension):
    """Train one bottleneck size and restore its best validation state."""
    torch.manual_seed(100 + latent_dimension)
    model = SpectralAutoencoder(latent_dimension).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-6,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        factor=0.5,
        patience=8,
        min_lr=1e-5,
    )
    loss_function = nn.MSELoss()

    history = {"train": [], "validation": []}
    best_validation_loss = np.inf
    best_state = None
    epochs_without_improvement = 0

    generator = torch.Generator().manual_seed(200 + latent_dimension)
    training_loader = DataLoader(
        training_set,
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )

    for epoch in range(maximum_epochs):
        model.train()
        training_loss_sum = 0.0

        for (batch,) in training_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            loss = loss_function(model(batch), batch)
            loss.backward()
            optimizer.step()
            training_loss_sum += loss.item() * batch.size(0)

        model.eval()
        validation_loss_sum = 0.0
        with torch.no_grad():
            for (batch,) in validation_loader:
                batch = batch.to(device)
                batch_loss = loss_function(model(batch), batch)
                validation_loss_sum += batch_loss.item() * batch.size(0)

        training_loss = training_loss_sum / len(training_set)
        validation_loss = validation_loss_sum / len(validation_set)
        history["train"].append(training_loss)
        history["validation"].append(validation_loss)
        scheduler.step(validation_loss)

        if validation_loss < best_validation_loss - 1e-6:
            best_validation_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= early_stopping_patience:
            break

    model.load_state_dict(best_state)
    history["epochs_completed"] = len(history["train"])
    history["best_validation_loss"] = best_validation_loss

    print(
        f"latent={latent_dimension:2d} | "
        f"epochs={history['epochs_completed']:3d} | "
        f"best validation MSE={best_validation_loss:.5f}"
    )
    return model, history


autoencoders = {}
histories = {}
training_start = time.perf_counter()

for latent_dimension in latent_dimensions:
    autoencoders[latent_dimension], histories[latent_dimension] = (
        train_autoencoder(latent_dimension)
    )

print(f"total training time = {time.perf_counter() - training_start:.1f} s")


In [ ]:
fig, ax = plt.subplots(figsize=(8.8, 4.8), constrained_layout=True)
colour_sequence = plt.cm.plasma(
    np.linspace(0.08, 0.88, len(latent_dimensions))
)

for colour, latent_dimension in zip(colour_sequence, latent_dimensions):
    validation_history = histories[latent_dimension]["validation"]
    ax.semilogy(
        np.arange(1, len(validation_history) + 1),
        validation_history,
        color=colour,
        label=f"d = {latent_dimension}",
    )

ax.set(
    xlabel="Training epoch",
    ylabel="Validation MSE in standardized flux",
    title="Validation histories for the deeper autoencoders",
)
ax.grid(alpha=0.22, which="both")
ax.legend(ncol=2)
plt.show()


## 6. Extract the compressed latent arrays and reconstructions

Calling `encode` stops exactly at the bottleneck. For a test set containing $N$ spectra, the stored code has shape $(N,d)$ and contains $Nd$ floating-point values. The decoder then turns those values back into standardized spectra, which we explicitly convert to physical normalized flux.


In [ ]:
test_tensor = spectrum_tensor(test_standard).to(device)
autoencoder_latents = {}
autoencoder_standard_reconstructions = {}
autoencoder_flux_reconstructions = {}

for latent_dimension, model in autoencoders.items():
    model.eval()
    with torch.no_grad():
        latent = model.encode(test_tensor)
        reconstruction = model.decode(latent)

    latent_array = latent.cpu().numpy()
    standard_reconstruction = reconstruction.cpu().numpy()[:, 0]
    flux_reconstruction = flux_mean + flux_std * standard_reconstruction

    autoencoder_latents[latent_dimension] = latent_array
    autoencoder_standard_reconstructions[latent_dimension] = (
        standard_reconstruction
    )
    autoencoder_flux_reconstructions[latent_dimension] = flux_reconstruction

example_latent = autoencoder_latents[reference_latent_dimension]
print(f"Original test array   : {test_flux.shape}")
print(f"Example latent array  : {example_latent.shape}")
print(
    "Per-spectrum compression ratio at d="
    f"{reference_latent_dimension}: "
    f"{number_of_pixels / reference_latent_dimension:.1f}:1"
)


## 7. Measure held-out reconstruction fidelity

We now use the ordinary root-mean-square error in continuum-normalized flux,

$$
\mathrm{RMSE}=\sqrt{\frac{1}{128}\sum_{j=1}^{128}
\left(\widehat F_j-F_j\right)^2}.
$$

RMSE is first calculated for every test spectrum. We average spectra within each held-out simulation and use the mean and standard deviation across simulations 20–26 for the final comparison. This prevents a simulation contributing more weight merely because it contains more selected skewers.


In [ ]:
def rmse_per_spectrum(truth, reconstruction):
    """Return one flux RMSE value for every spectrum."""
    return np.sqrt(np.mean((reconstruction - truth) ** 2, axis=1))


def average_by_test_simulation(values):
    """Average per-spectrum values separately in each held-out box."""
    return np.array(
        [
            values[test_labels == simulation].mean()
            for simulation in test_simulations
        ]
    )


pca_reconstructions = {
    dimension: pca_reconstruct(test_flux, dimension)
    for dimension in pca_dimensions
}

autoencoder_rmse_by_simulation = {
    dimension: average_by_test_simulation(
        rmse_per_spectrum(test_flux, reconstruction)
    )
    for dimension, reconstruction in autoencoder_flux_reconstructions.items()
}
pca_rmse_by_simulation = {
    dimension: average_by_test_simulation(
        rmse_per_spectrum(test_flux, reconstruction)
    )
    for dimension, reconstruction in pca_reconstructions.items()
}

print("Mean test RMSE in normalized flux")
print(" dimension     PCA       autoencoder")
for dimension in code_dimensions:
    pca_mean_rmse = pca_rmse_by_simulation[dimension].mean()
    ae_mean_rmse = autoencoder_rmse_by_simulation[dimension].mean()
    print(f" {dimension:9d}   {pca_mean_rmse:7.4f}      {ae_mean_rmse:7.4f}")


In [ ]:
# A compact numerical check before inspecting individual spectra.
pca_mean_rmse = np.array(
    [pca_rmse_by_simulation[d].mean() for d in code_dimensions]
)
ae_mean_rmse = np.array(
    [autoencoder_rmse_by_simulation[d].mean() for d in code_dimensions]
)

print("PCA test RMSE decreases monotonically:", np.all(np.diff(pca_mean_rmse) <= 0))
print("AE test RMSE decreases monotonically: ", np.all(np.diff(ae_mean_rmse) <= 0))


## 8. How increasing dimension restores spectral information

A single error number cannot show *which* information has returned. The panels below follow the same held-out spectrum at dimensions $d=1,4,16,64$. Each column adds capacity; the top and bottom rows show PCA and the autoencoder at exactly the same dimension.

Look for three changes: recovery of broad transmission trends, separation of neighbouring troughs, and restoration of narrow absorption structure.


In [ ]:
displayed_dimensions = [1, 4, 16, 64]
original_flux = test_flux[representative]

fig, axes = plt.subplots(
    2,
    len(displayed_dimensions),
    figsize=(16, 6.8),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

for column, dimension in enumerate(displayed_dimensions):
    method_rows = [
        ("PCA", pca_reconstructions[dimension][representative], BLUE),
        (
            "Autoencoder",
            autoencoder_flux_reconstructions[dimension][representative],
            ORANGE,
        ),
    ]

    for row, (method, reconstruction, colour) in enumerate(method_rows):
        ax = axes[row, column]
        spectrum_rmse = np.sqrt(
            np.mean((reconstruction - original_flux) ** 2)
        )
        ax.step(
            x,
            original_flux,
            where="mid",
            color=LIGHT_GREY,
            linewidth=1.0,
            label="Original" if column == 0 else None,
        )
        ax.plot(
            x,
            reconstruction,
            color=colour,
            linewidth=1.6,
            label=method if column == 0 else None,
        )
        ax.text(
            0.04,
            0.06,
            f"RMSE = {spectrum_rmse:.3f}",
            transform=ax.transAxes,
            fontsize=8.5,
            bbox={
                "facecolor": "white",
                "alpha": 0.82,
                "edgecolor": "none",
                "pad": 2,
            },
        )
        ax.set(xlim=(0, box_size), ylim=(-0.05, 1.05))
        ax.grid(axis="x", alpha=0.18)

        if row == 0:
            ax.set_title(f"d = {dimension}", fontweight="bold")
        if column == 0:
            ax.set_ylabel(f"{method}\nflux")
        if row == 1:
            ax.set_xlabel(r"$x$ [$h^{-1}$ cMpc]")

axes[0, 0].legend(loc="upper right", fontsize=8)
axes[1, 0].legend(loc="upper right", fontsize=8)
fig.suptitle(
    "More stored dimensions progressively restore Lyα-forest structure",
    fontsize=15,
    fontweight="semibold",
)
plt.show()


## 9. Matched-fidelity comparison

We take the mean test RMSE of the eight-dimensional autoencoder as a transparent reference and find the smallest tested PCA representation whose mean error is no larger. Because both methods now use the identical dimension grid $1,2,4,8,16,32,64$, this comparison does not benefit either method through denser sampling.

If PCA reaches the target with fewer coefficients, the data do not support an autoencoder compression advantage under this setup; the calculation reports that outcome directly.


In [ ]:
reference_error = autoencoder_rmse_by_simulation[
    reference_latent_dimension
].mean()
qualifying_pca_dimensions = [
    dimension
    for dimension in pca_dimensions
    if pca_rmse_by_simulation[dimension].mean() <= reference_error
]
matched_pca_dimension = (
    min(qualifying_pca_dimensions)
    if qualifying_pca_dimensions
    else max(pca_dimensions)
)
pca_reached_target = bool(qualifying_pca_dimensions)

print(
    f"Reference autoencoder: d={reference_latent_dimension}, "
    f"mean test RMSE={reference_error:.4f}"
)
if pca_reached_target:
    print(
        "Smallest tested PCA code reaching this RMSE: "
        f"d={matched_pca_dimension}"
    )
else:
    print(
        "PCA did not reach this RMSE with up to "
        f"d={matched_pca_dimension}."
    )

if matched_pca_dimension > reference_latent_dimension:
    print("Result: the autoencoder uses the smaller per-spectrum code.")
elif matched_pca_dimension == reference_latent_dimension:
    print("Result: both methods require the same tested code size.")
else:
    print("Result: PCA uses the smaller code for this experiment.")


In [ ]:
original_value_count = test_flux.size
ae_value_count = autoencoder_latents[reference_latent_dimension].size
pca_coefficients = pca_encode(test_flux, matched_pca_dimension)
pca_value_count = pca_coefficients.size

print("Compressed test arrays, assuming float32 storage")
print(
    f"Original    : {original_value_count:7d} values, "
    f"{4 * original_value_count / 1024:.1f} KiB"
)
print(
    f"Autoencoder : {ae_value_count:7d} values, "
    f"{4 * ae_value_count / 1024:.1f} KiB"
)
print(
    f"PCA         : {pca_value_count:7d} values, "
    f"{4 * pca_value_count / 1024:.1f} KiB"
)

fig, ax = plt.subplots(figsize=(7.6, 4.4), constrained_layout=True)
labels = [
    "Original flux",
    f"AE latent\n(d={reference_latent_dimension})",
    f"PCA coefficients\n(d={matched_pca_dimension})",
]
values_per_spectrum = [
    number_of_pixels,
    reference_latent_dimension,
    matched_pca_dimension,
]
bars = ax.bar(
    labels,
    values_per_spectrum,
    color=[BLACK, ORANGE, BLUE],
    width=0.68,
)
ax.bar_label(bars, padding=3)
ax.set(
    ylabel="Stored float values per spectrum",
    title="Representation size at matched reconstruction fidelity",
    ylim=(0, number_of_pixels * 1.12),
)
ax.grid(axis="y", alpha=0.2)
plt.show()


## 10. Inspect the matched-fidelity reconstructions

The residual panel is essential: two reconstructions may look similar while differing systematically in narrow or saturated absorption features. Both curves below use the dimensions selected by the matched-fidelity calculation above.


In [ ]:
original_flux = test_flux[representative]
ae_flux = autoencoder_flux_reconstructions[
    reference_latent_dimension
][representative]
pca_flux = pca_reconstructions[matched_pca_dimension][representative]

fig, axes = plt.subplots(
    2,
    1,
    figsize=(10.8, 6.4),
    sharex=True,
    constrained_layout=True,
    gridspec_kw={"height_ratios": [1.7, 1]},
)
axes[0].step(
    x,
    original_flux,
    where="mid",
    color=BLACK,
    linewidth=1.2,
    label="Original",
)
axes[0].plot(
    x,
    ae_flux,
    color=ORANGE,
    linewidth=2.0,
    label=f"Autoencoder (d={reference_latent_dimension})",
)
axes[0].plot(
    x,
    pca_flux,
    color=BLUE,
    linewidth=1.7,
    label=f"PCA (d={matched_pca_dimension})",
)
axes[0].set(
    ylabel="Transmitted flux",
    title="Matched-fidelity reconstruction of one held-out spectrum",
    ylim=(-0.04, 1.05),
)
axes[0].legend(ncol=3)
axes[0].grid(axis="x", alpha=0.18)

axes[1].plot(
    x,
    ae_flux - original_flux,
    color=ORANGE,
    label="Autoencoder residual",
)
axes[1].plot(
    x,
    pca_flux - original_flux,
    color=BLUE,
    label="PCA residual",
)
axes[1].axhline(0, color=GREY, linewidth=0.9)
axes[1].set(
    xlabel=r"Distance $x$ [$h^{-1}$ cMpc]",
    ylabel="Reconstruction − truth",
    xlim=(0, box_size),
)
axes[1].legend()
axes[1].grid(axis="x", alpha=0.18)
plt.show()


## 11. Code size is not the whole storage cost

The comparison above concerns the **per-spectrum compressed array**. A reusable compressor must also store shared information:

- PCA stores one mean spectrum and its component basis;
- the autoencoder stores all learned encoder and decoder parameters.

The deeper neural model is substantially larger than the PCA basis. A smaller latent array becomes a total-storage advantage only when many spectra share the same trained network.


In [ ]:
ae_parameter_count = sum(
    parameter.numel()
    for parameter in autoencoders[reference_latent_dimension].parameters()
)
pca_decoder_value_count = number_of_pixels * (
    matched_pca_dimension + 1
)

print("Shared decoder/model storage, assuming float32")
print(
    f"Autoencoder parameters : {ae_parameter_count:,} values "
    f"({4 * ae_parameter_count / 1024:.1f} KiB)"
)
print(
    f"PCA mean + basis       : {pca_decoder_value_count:,} values "
    f"({4 * pca_decoder_value_count / 1024:.1f} KiB)"
)

if matched_pca_dimension > reference_latent_dimension:
    extra_model_values = ae_parameter_count - pca_decoder_value_count
    saved_values_per_spectrum = (
        matched_pca_dimension - reference_latent_dimension
    )
    if extra_model_values > 0 and saved_values_per_spectrum > 0:
        break_even_spectra = int(
            np.ceil(extra_model_values / saved_values_per_spectrum)
        )
        print(
            "Approximate float-count break-even dataset size: "
            f"{break_even_spectra:,} spectra"
        )


## 12. Final comparison: RMSE versus compressed dimension

This is the central benchmark. Both curves use the same seven dimensions and the same held-out spectra. Markers show the mean of the per-simulation RMSE values; error bars show their standard deviation across the seven test simulations. The upper axis translates dimension into the per-spectrum compression ratio $128/d$.


In [ ]:
pca_rmse_mean = np.array(
    [pca_rmse_by_simulation[d].mean() for d in code_dimensions]
)
pca_rmse_std = np.array(
    [pca_rmse_by_simulation[d].std(ddof=1) for d in code_dimensions]
)
ae_rmse_mean = np.array(
    [autoencoder_rmse_by_simulation[d].mean() for d in code_dimensions]
)
ae_rmse_std = np.array(
    [
        autoencoder_rmse_by_simulation[d].std(ddof=1)
        for d in code_dimensions
    ]
)

fig, ax = plt.subplots(figsize=(9, 5.6), constrained_layout=True)
ax.errorbar(
    code_dimensions,
    pca_rmse_mean,
    yerr=pca_rmse_std,
    color=BLUE,
    marker="s",
    markersize=6,
    linewidth=2.2,
    capsize=3,
    label="PCA",
)
ax.errorbar(
    code_dimensions,
    ae_rmse_mean,
    yerr=ae_rmse_std,
    color=ORANGE,
    marker="o",
    markersize=6,
    linewidth=2.2,
    capsize=3,
    label="Deep autoencoder",
)
ax.set(
    xscale="log",
    yscale="log",
    xlabel="Compressed dimension d [stored values per spectrum]",
    ylabel="Held-out flux RMSE",
    title="Reconstruction error versus compressed dimension",
)
ax.set_xticks(
    code_dimensions,
    labels=[str(value) for value in code_dimensions],
)
ax.grid(alpha=0.24, which="both")
ax.legend()

compression_axis = ax.secondary_xaxis("top")
compression_axis.set_xscale("log")
compression_axis.set_xticks(
    code_dimensions,
    labels=[f"{number_of_pixels // d}:1" for d in code_dimensions],
)
compression_axis.set_xlabel("Per-spectrum compression ratio")

plt.show()


## Conclusions

- The PCA implementation is an exact truncated SVD: its basis is orthonormal, its training error decreases monotonically, and the full-rank inverse transform reconstructs to numerical precision.
- PCA is therefore a strong and trustworthy linear baseline; weak recovery at very small $d$ reflects the limitation of a single linear subspace, not an incorrect transpose or inverse transform.
- The deeper autoencoder uses four encoder and four decoder stages, with two convolutions per stage and no skip connections around the bottleneck.
- Progressive reconstruction panels reveal which broad and narrow features return as $d$ increases.
- The overlaid held-out RMSE curve—not a single preferred example—determines whether nonlinear compression is advantageous for this dataset.
- Per-spectrum latent size and total storage are different questions because the neural network has a larger shared model overhead.


## Exercises

1. Replace GELU with a purely linear activation. Compare the resulting error curve with PCA and explain the connection.
2. Add Gaussian pixel noise and distinguish ordinary autoencoding from denoising autoencoding by changing the input and target arrays.
3. Compare RMSE separately in high-transmission pixels ($F>0.8$) and strong-absorption pixels ($F<0.2$).
4. Train at one instrumental resolution and test at another to measure transfer across spectral resolution.
5. Plot the two-dimensional latent code for $d=2$, coloured by mean flux or effective optical depth.
6. Change average pooling to max pooling. Which choice better preserves broad transmission structure, and which better preserves narrow extrema?
7. Repeat training with three random initializations at one dimension and compare initialization scatter with the test-simulation error bars.


## Final take-away

> PCA stores coordinates on the best linear subspace and provides a mathematically exact baseline. A deep autoencoder may represent a curved spectral manifold more efficiently, but that claim is justified only when the two methods use the same dimensions, the same held-out spectra, and an overlaid reconstruction-error curve. The progressive spectral panels show what the RMSE curve means physically.
